In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install openai datasets tqdm
!pip install json

ERROR: Could not find a version that satisfies the requirement json (from versions: none)
ERROR: No matching distribution found for json


In [ ]:
# 1. 환경 설정
import os, json, random
from tqdm import tqdm
from datasets import Dataset, DatasetDict
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = MY_KEY  # 또는 getpass로 입력
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [ ]:
# 2. context 데이터 로드
# context 파일 예시: [{"text": "...", "title": "...", "document_id": "...", "url": "..."}]
with open("/content/drive/MyDrive/Colab Notebooks/odqa_data/data/wikipedia_documents.json", "r", encoding="utf-8") as f:
    contexts = json.load(f)

print("문서 개수:", len(contexts))


문서 개수: 60613


In [ ]:
import re

def is_valid_context(text):
    """비정상 텍스트(파일명, 경로, 태그 등) 필터링"""
    if not isinstance(text, str):
        return False
    # 너무 짧거나 공백뿐인 경우
    if len(text.strip()) < 10:
        return False
    # 파일 경로, 확장자, 태그 등 포함 시 제외
    invalid_patterns = [
        r'\bhttps?://',         # URL
        r'\.jpg|\.png|\.pdf',   # 파일 확장자
        r'[A-Za-z0-9_/\\-]+\.(txt|csv|xlsx|json)',  # 파일명 패턴
        r'<[^>]+>',             # HTML 태그
    ]
    for pat in invalid_patterns:
        if re.search(pat, text):
            return False
    return True

In [ ]:
# @title
import json, re, random, statistics
from collections import Counter

path = "/content/drive/MyDrive/Colab Notebooks/odqa_data/data/wikipedia_documents.json"

with open(path, "r", encoding="utf-8") as f:
    contexts = json.load(f)

# dict → list 변환
if isinstance(contexts, dict):
    contexts = list(contexts.values())

print(f"[요약] 타입=list, 문서 수={len(contexts)}")

all_keys = Counter()
for d in contexts:
    all_keys.update(d.keys())

print("\n[키 분포 top]")
for k, v in all_keys.most_common():
    print(f" - {k}: {v}/{len(contexts)}")

required = ["text", "title", "document_id", "url"]
missing_by_doc = {k: [] for k in required}
empty_by_doc = {k: [] for k in required}

for i, d in enumerate(contexts):
    for k in required:
        if k not in d:
            missing_by_doc[k].append(i)
        else:
            val = d[k]
            if val is None or (isinstance(val, str) and val.strip() == ""):
                empty_by_doc[k].append(i)

print("\n[필수 키 결측 인덱스 수]")
for k in required:
    print(f" - {k}: 결측 {len(missing_by_doc[k])}, 비어있음 {len(empty_by_doc[k])}")

# 3) document_id 중복 점검
ids = [d.get("document_id") for d in contexts if "document_id" in d]
dup_ids = [k for k, c in Counter(ids).items() if c > 1]
print(f"\n[document_id] 고유 개수={len(set(ids))}, 중복 개수={len(dup_ids)}")
if dup_ids:
    print(" 예시 중복 ID 5개까지:", dup_ids[:5])

# 4) text 길이 통계
lengths = [len(d.get("text","")) for d in contexts]
def qtile(arr, q):
    arr_sorted = sorted(arr)
    idx = int((len(arr_sorted)-1)*q)
    return arr_sorted[idx] if arr_sorted else None

print("\n[text 길이(문자 수) 통계]")
print(f" - min={min(lengths) if lengths else None}")
print(f" - 25%={qtile(lengths, 0.25)}  median={qtile(lengths, 0.5)}  75%={qtile(lengths, 0.75)}")
print(f" - mean={round(statistics.mean(lengths),2) if lengths else None}")
print(f" - max={max(lengths) if lengths else None}")

# 5) URL 형식 간단 점검
url_re = re.compile(r"^https?://")
bad_urls = [i for i, d in enumerate(contexts) if "url" in d and not url_re.match(str(d["url"]))]

print(f"\n[url 형식 점검] 형식 비일치 건수={len(bad_urls)}")
if bad_urls:
    print(" 예시 인덱스 5개까지:", bad_urls[:5])

# 6) 샘플 문서 2개 출력(키/값 미리보기)
def preview(d, max_chars=300):
    out = {}
    for k, v in d.items():
        if isinstance(v, str) and len(v) > max_chars:
            out[k] = v[:max_chars].rstrip() + " ... (truncated)"
        else:
            out[k] = v
    return out

print("\n[샘플 2건 미리보기]")
for idx in [0, min(1, len(contexts)-1)] if len(contexts) >= 2 else [0]:
    print(f"\n- index={idx}")
    for k, v in preview(contexts[idx]).items():
        print(f"  {k}: {v}")

# 7) 랜덤 샘플 1건(빠른 육안 확인용)
if contexts:
    ridx = random.randrange(len(contexts))
    print(f"\n[랜덤 샘플] index={ridx}")
    for k, v in preview(contexts[ridx]).items():
        print(f"  {k}: {v}")


[요약] 타입=list, 문서 수=60613

[키 분포 top]
 - text: 60613/60613
 - corpus_source: 60613/60613
 - url: 60613/60613
 - domain: 60613/60613
 - title: 60613/60613
 - author: 60613/60613
 - html: 60613/60613
 - document_id: 60613/60613

[필수 키 결측 인덱스 수]
 - text: 결측 0, 비어있음 0
 - title: 결측 0, 비어있음 0
 - document_id: 결측 0, 비어있음 0
 - url: 결측 0, 비어있음 56059

[document_id] 고유 개수=60613, 중복 개수=0

[text 길이(문자 수) 통계]
 - min=184
 - 25%=414  median=577  75%=857
 - mean=755.57
 - max=46099

[url 형식 점검] 형식 비일치 건수=60613
 예시 인덱스 5개까지: [0, 1, 2, 3, 4]

[샘플 2건 미리보기]

- index=0
  text: 이 문서는 나라 목록이며, 전 세계 206개 나라의 각 현황과 주권 승인 정보를 개요 형태로 나열하고 있다.

이 목록은 명료화를 위해 두 부분으로 나뉘어 있다.

# 첫 번째 부분은 바티칸 시국과 팔레스타인을 포함하여 유엔 등 국제 기구에 가입되어 국제적인 승인을 널리 받았다고 여기는 195개 나라를 나열하고 있다.
# 두 번째 부분은 일부 지역의 주권을 사실상 (데 팍토) 행사하고 있지만, 아직 국제적인 승인을 널리 받지 않았다고 여기는 11개 나라를 나열하고 있다.

두 목록은 모두 가나다 순이다.

일부 국가의 경우 국가로서 ... (truncated)
  corpus_source: 위키피디아
  url: TODO
  domain: None
  title: 나라 목록
  author: None
  html: None
  document_id: 0

- index=1
  

In [ ]:
from datasets import load_from_disk

# 이미 로드된 상태라면 이 부분 생략 가능
train_dataset = load_from_disk("/content/drive/MyDrive/Colab Notebooks/odqa_data/data/train_dataset")

# train, validation 추출
train_ids = set(train_dataset["train"]["document_id"])
val_ids = set(train_dataset["validation"]["document_id"])
used_ids = train_ids.union(val_ids)

print(f"train+val에서 사용된 document_id 개수: {len(used_ids)}")

# contexts dict → list 변환 (앞서처럼)
with open("/content/drive/MyDrive/Colab Notebooks/odqa_data/data/wikipedia_documents.json", "r", encoding="utf-8") as f:
    contexts = json.load(f)
if isinstance(contexts, dict):
    contexts = list(contexts.values())

print(f"원본 context 문서 수: {len(contexts)}")

# 필터링
filtered_contexts = [c for c in contexts if c["document_id"] not in used_ids]

print(f"필터링 후 context 문서 수: {len(filtered_contexts)} (제외된 문서 {len(contexts)-len(filtered_contexts)})")

# 결과 확인
print("\n예시 2건:")
for c in filtered_contexts[:2]:
    print(f"- id={c['document_id']} / title={c['title'][:50]}")

# 필요하다면 저장
with open("/content/drive/MyDrive/Colab Notebooks/odqa_data/data/wikipedia_documents_filtered.json", "w", encoding="utf-8") as f:
    json.dump(filtered_contexts, f, ensure_ascii=False, indent=2)


train+val에서 사용된 document_id 개수: 3504
원본 context 문서 수: 60613
필터링 후 context 문서 수: 57109 (제외된 문서 3504)

예시 2건:
- id=0 / title=나라 목록
- id=1 / title=나라 목록


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import json

# 1️⃣ train context / unused context 불러오기
train_texts = [c['context'] for c in train_dataset['train']]
unused_texts = [c['text'] for c in filtered_contexts]

print(f"Train 문서 수: {len(train_texts)}")
print(f"Unused 문서 수: {len(unused_texts)}")

# 2️⃣ TF-IDF 벡터화
vectorizer = TfidfVectorizer(
    max_features=20000,     # 상위 2만 단어만 사용
    stop_words=None,        # 한국어의 경우 불용어 직접 지정 가능
    ngram_range=(1,2),      # unigram + bigram 고려
)
X_train = vectorizer.fit_transform(train_texts)
X_unused = vectorizer.transform(unused_texts)

print("TF-IDF 행렬 크기:", X_train.shape, X_unused.shape)

# 3️⃣ 코사인 유사도 계산
# 각 unused 문서마다 가장 유사한 train 문서와의 최대 유사도 구함
similarity_matrix = cosine_similarity(X_unused, X_train)
max_similarities = similarity_matrix.max(axis=1)

# 4️⃣ 결과 정리
avg_sim = np.mean(max_similarities)
print(f"평균 최대 유사도: {avg_sim:.3f}")

# threshold 이상 → 제외, 그 미만만 남김
threshold = 0.20
selected_idx = np.where(max_similarities < threshold)[0]
filtered_for_augmentation = [filtered_contexts[i] for i in selected_idx]

print(f"임계값 {threshold} 미만 문서 수: {len(filtered_for_augmentation)} / {len(filtered_contexts)}")

# 예시 확인
low_k = np.argsort(max_similarities)[:5]  # 유사도 가장 낮은 5건
print("\n[유사도 낮은 문서 5건 예시]")
for i in low_k:
    print(f"- index={i}, 유사도={max_similarities[i]:.3f}, title={filtered_contexts[i]['title'][:50]}")

# 6️⃣ (선택) 저장
with open("/content/drive/MyDrive/Colab Notebooks/odqa_data/data/filtered_contexts_by_similarity.json", "w", encoding="utf-8") as f:
    json.dump(filtered_for_augmentation, f, ensure_ascii=False, indent=2)


Train 문서 수: 3952
Unused 문서 수: 57109
TF-IDF 행렬 크기: (3952, 20000) (57109, 20000)
평균 최대 유사도: 0.190
임계값 0.2 미만 문서 수: 39942 / 57109

[유사도 낮은 문서 5건 예시]
- index=8516, 유사도=0.000, title=오스트리아의 국가
- index=674, 유사도=0.000, title=포유류의 목록
- index=6803, 유사도=0.000, title=하노이의 탑
- index=8511, 유사도=0.000, title=아르차흐 공화국의 국가
- index=35886, 유사도=0.000, title=핫식스 리그 오브 레전드 챔피언스 서머 2014


In [ ]:
import re

def is_valid_context(text):
    """비정상 텍스트(파일명, 경로, 태그 등) 필터링"""
    if not isinstance(text, str):
        return False
    # 너무 짧거나 공백뿐인 경우
    if len(text.strip()) < 10:
        return False
    # 파일 경로, 확장자, 태그 등 포함 시 제외
    invalid_patterns = [
        r'\bhttps?://',         # URL
        r'\.jpg|\.png|\.pdf',   # 파일 확장자
        r'[A-Za-z0-9_/\\-]+\.(txt|csv|xlsx|json)',  # 파일명 패턴
        r'<[^>]+>',             # HTML 태그
    ]
    for pat in invalid_patterns:
        if re.search(pat, text):
            return False
    return True

In [ ]:
# 저장된 파일 로드
input_path = "/content/drive/MyDrive/Colab Notebooks/odqa_data/data/filtered_contexts_by_similarity.json"
with open(input_path, "r", encoding="utf-8") as f:
    filtered_for_augmentation = json.load(f)

print(f"로드된 문서 수: {len(filtered_for_augmentation)}")

# 필터링 적용
cleaned_contexts = [
    c for c in filtered_for_augmentation
    if is_valid_context(c.get("text", ""))  # key가 'text'일 가능성 높음
]

print(f"정제 후 문서 수: {len(cleaned_contexts)} / {len(filtered_for_augmentation)}")


로드된 문서 수: 39942
정제 후 문서 수: 38171 / 39942


In [ ]:
output_path = "/content/drive/MyDrive/Colab Notebooks/odqa_data/data/filtered_contexts_cleaned.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(cleaned_contexts, f, ensure_ascii=False, indent=2)

print("정상 데이터만 저장 완료:", output_path)


정상 데이터만 저장 완료: /content/drive/MyDrive/Colab Notebooks/odqa_data/data/filtered_contexts_cleaned.json


In [ ]:
# 중복 없이 5000개 무작위 샘플링
sample_size = 5000
sampled_contexts = random.sample(cleaned_contexts, sample_size)

print(f"총 {len(cleaned_contexts)}개 중 {len(sampled_contexts)}개 샘플링 완료")

총 38171개 중 5000개 샘플링 완료


In [ ]:
# 3. GPT-4o mini 호출 함수
def generate_qa_from_context(context_obj, n=1):
    """GPT-4o mini로 context 기반 QA 생성"""
    context_text = context_obj["text"].strip()
    title = context_obj.get("title", "")
    doc_id = context_obj.get("document_id", "")

    prompt = f"""
당신은 한국어 Extractive QA 데이터셋 생성 전문가입니다.
다음 문단을 읽고 사실 기반 질문 {n}개와 각 질문의 답변을 JSON 형식으로 생성하십시오.
그 외의 설명, 서문, 마크다운 표기, 따옴표 밖 텍스트는 절대 출력하지 마십시오.

조건:
- answer는 반드시 context에 실제로 존재하는 연속된 문자열이어야 합니다.
- answer는 문장 전체가 아니라, 질문에 직접적으로 대응되는 핵심 구절만 사용하십시오.
- answer_start는 context의 첫 번째 문자를 0으로 하는 문자 단위(0-based) 인덱스로 표기하십시오.
- question은 answer를 그대로 정답으로 가질 수 있어야 하며, answer를 포함하지 않아야 합니다.
- question은 context의 내용을 모르는 독자도 이해할 수 있도록 완전한 문장으로 작성하십시오.
- question에는 ‘이 영화’, ‘이 인물’, ‘이 밴드’, ‘그 사람’, ‘이 사건’ 등
  대명사나 지시어를 사용하지 마십시오.
- question은 항상 context 내부의 내용을 기반으로 하며, 외부 지식에 의존하지 않아야 합니다.
- answer는 title이면 안됩니다.
- question의 유형은 다양하게 하시오.
- 출력은 JSON 배열 형식으로만 작성합니다.


예시 형식:
[
  {{"question": "대통령을 포함한 미국의 행정부 견제권을 갖는 국가 기관은?", "answer": "하원", "answer_start": [235]}}
  ...
]

title: {title}
context : {context_text}
"""
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "당신은 QA 데이터 생성 전문가입니다."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.5
        )

        output = response.choices[0].message.content.strip()
        # 모델 출력 파싱
        data = json.loads(output)

        qa_list = []
        for i, qa in enumerate(data):
            q = qa.get("question", "").strip()
            a = qa.get("answer", "").strip()
            if not q or not a:
                continue

            start_idx = context_text.find(a)
            if start_idx == -1:
                start_idx = None

            qa_item = {
                "title": title,
                "context": context_text,  # 후처리에서 context 삽입
                "question": q,
                "id": f"{doc_id}_q{i}",
                "answers": {
                    "answer_start": [start_idx] if start_idx is not None else [],
                    "text": [a]
                },
                "document_id": doc_id
            }
            qa_list.append(qa_item)

        return qa_list

    except Exception as e:
        print(f"[오류] {doc_id} 처리 중 예외 발생: {e}")
        return []


In [ ]:
# 4. 데이터셋 생성 루프

augmented_data = []

# 예: 필터링된 context 중 상위 100개만 증강
for c in tqdm(sampled_contexts[:], desc="QA 생성 중"):
    qa_list = generate_qa_from_context(c, n=1)
    augmented_data.extend(qa_list)

print(f"생성된 QA 개수: {len(augmented_data)}")


QA 생성 중:  23%|██▎       | 1137/5000 [2:31:05<10:05:41,  9.41s/it]

[오류] 53029 처리 중 예외 발생: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-YW9REqrlnXc08xl1r0qDbXhB on requests per day (RPD): Limit 10000, Used 10000, Requested 1. Please try again in 8.64s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}


QA 생성 중:  36%|███▌      | 1785/5000 [4:04:25<9:49:45, 11.01s/it]

[오류] 2639 처리 중 예외 발생: Invalid \escape: line 2 column 94 (char 95)


QA 생성 중:  46%|████▌     | 2302/5000 [5:18:38<6:07:44,  8.18s/it]

...

[오류] 3764 처리 중 예외 발생: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-YW9REqrlnXc08xl1r0qDbXhB on requests per day (RPD): Limit 10000, Used 10000, Requested 1. Please try again in 8.64s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}


QA 생성 중: 100%|█████████▉| 4997/5000 [11:41:41<00:26,  8.84s/it]

[오류] 23660 처리 중 예외 발생: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o-mini in organization org-YW9REqrlnXc08xl1r0qDbXhB on requests per day (RPD): Limit 10000, Used 10000, Requested 1. Please try again in 8.64s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}


QA 생성 중: 100%|██████████| 5000/5000 [11:42:11<00:00,  8.43s/it]

생성된 QA 개수: 4971


In [ ]:
import pandas as pd

# pandas → Dataset 변환
df = pd.DataFrame(augmented_data)

# Dataset으로 변환
aug_dataset = Dataset.from_pandas(df)

# 구조 확인
print(aug_dataset)
print(aug_dataset[0])


Dataset({
    features: ['title', 'context', 'question', 'id', 'answers', 'document_id'],
    num_rows: 4971
})
{'title': '한솔로지스틱스', 'context': '한솔로지스틱스 주식회사(Hansol Logistics Co., Ltd.) 는, 대한민국의 화물운송 중개, 대리 및 관련서비스업을 영위하는 한솔그룹의 계열사이다. 한솔로지스틱스(주)의 설립일과 그 명칭은 1994년 6월에 한솔유통(주)라는 사명으로 설립되었다. 1996년 10월엔 유가증권시장 상장사인 영우통상을 인수하였다. 1997년 2월 사명을 한솔CSN으로 변경하였다. 1998년 9월엔 철도청으로부터 철탑산업훈장을 수상했다. 2000년 1월 군산항컨테이너부두 운영을 개시하였고, 2006년 5월 ISO9001과 ISO14001 인증을 한국표준협회로부터 받았다. 2007년 4월엔, 부산 신항 합작법인 FCL 출범과 군산항 물동량증대 협약을 체결하였고 2007년 7월에는 중국법인을 상해와 천진에 설립하였다. 2007년 9월과 2008년 9월엔 미주법인 설립과 인도법인을 각각 설립했다. 2011년 12월엔 관세청으로부터 AEO 인증을 받았고 2012년 1월 본사를 서울특별시 중구 을지로 100으로 이전하였으며 2012년 8월 말레이시아에 법인을 설립하였다. 2013년 5월 신탄진 CY를 오픈했고 2014년 5월 사명을 한솔CSN에서 현재의 사명인 한솔로지스틱스(주)로 변경하였다. 2015년 7월엔 멕시코에 법인을 설립하였다. 한솔로지스틱스(주)의 현재 대표이사는 민병규 대표이사 사장이다.', 'question': '한솔로지스틱스 주식회사가 설립된 해는 언제인가요?', 'id': '45613_q0', 'answers': {'answer_start': [113], 'text': ['1994년 6월']}, 'document_id': 45613}


In [ ]:
import pandas as pd
from datasets import Dataset

# 1️⃣ pandas DataFrame으로 변환
df = pd.DataFrame(augmented_data)

# 2️⃣ Hugging Face Dataset으로 변환
aug_dataset = Dataset.from_pandas(df)

# 3️⃣ Dataset 저장
save_path = "/content/drive/MyDrive/Colab Notebooks/odqa_data/data/augmented_dataset2"
aug_dataset.save_to_disk(save_path)

print(f"데이터셋이 '{save_path}' 경로에 저장되었습니다.")

Saving the dataset (0/1 shards):   0%|          | 0/4971 [00:00<?, ? examples/s]

데이터셋이 '/content/drive/MyDrive/Colab Notebooks/odqa_data/data/augmented_dataset2' 경로에 저장되었습니다.


In [ ]:
import random
import pandas as pd

# 예: 30개만 샘플링
sample_size = 30
indices = random.sample(range(len(aug_dataset)), sample_size)
sampled_dataset = aug_dataset.select(indices)

print(f"총 {len(aug_dataset)}개 중 {len(sampled_dataset)}개를 추출했습니다.")
print(sampled_dataset[0])

총 4971개 중 30개를 추출했습니다.
{'title': '태권도', 'context': '1945년 일제 해방 이후 국내에 여러 개의 무술 도장이 생기게 된다. 그 중 소위 \'5대관\'(청도관, 송무관, 무덕관, 지도관, 창무관)이 가장 유명하였는데, 이 도장들이 분화하여 생긴 9개관이 1960년대에 합쳐져서 현대 태권도의 모체가 된다. 최초의 태권도장인 청도관은 이원국에 의해 설립됐는데, 그는 어렸을 때 서울 안국동에서 택견을 수련했고날짜=2017-05-18, 이후 일본에서 공수도를 배우고, 중국에서는 쿵푸를 수련했다. 당시 최대의 태권도장이던 무덕관(1953년과 1970년 사이에 전체 태권도 수련자의 약 75%가 무덕관에서 배웠다.)은 황기에 의해서 설립됐는데, 그는 어릴 때 택견을 배우고날짜=2017-05-18, 중국에서 태극권과 쿵푸를 배웠다. (그는 직접 공수도를 배운 적은 없다. ) 이후 한국의 무예서 《무예도보통지》를 연구하여 무덕관의 기술을 완성시킨다.\n\n한편 군인이었던 최홍희는 군을 중심으로한 오도관에서 무술을 보급했다. 그는 여려서 택견을 배우고  , 일본 중앙대학을 다니면서 공수도를 배운 뒤, 군에서 복무하며 군대격투기로 공수도를 지도하였다. 1954년 이승만 대통령이 육군의 공수도 시범을 관람한 후 "어린 시절 본 택견과 비슷하다"라고 언급하였고, 당시 육군 장성 최홍희가 택견을 태권으로 바꾸고, 여기에 도를 합하여 1955년 태권도라는 명칭을 탄생시켰다.  최홍희가 총재를 맡았던 국제 태권도 연맹(ITF)에서는 최홍희를 태권도의 창시자로 보고 있으나, 그리고, 1973년에는 국기원이 건립되며 태권도는 한국 고유의 무도로서 자리를 잡아갔으며, 사범 개인의 차원에서 이루어지던 해외 진출이 국가와 연맹의 차원에서 본격적으로 이루어졌다. 1973년 최홍희가 정치적인 이유로 캐나다로 망명하여 ITF 본거지를 토론토로 옮기자, 대한민국 정부는 대한 태권도 협회를 중심으로 새로 세계 태권도 연맹(WTF)을 창립하여 태권도 보급에 나선다. I

In [ ]:
# aug_dataset 또는 filtered_dataset 중 하나 사용
df = aug_dataset.to_pandas()

save_path = "/content/drive/MyDrive/Colab Notebooks/odqa_data/data/augmented_dataset2.json"

# JSON으로 저장 (한글 깨짐 방지 + 보기 좋게 들여쓰기)
df.to_json(
    save_path,
    orient="records",
    force_ascii=False,
    indent=2
)

print(f"JSON 파일로 저장 완료: {save_path}")


JSON 파일로 저장 완료: /content/drive/MyDrive/Colab Notebooks/odqa_data/data/augmented_dataset2.json
